In [1]:

import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw")

df_sales = pd.read_csv(DATA_PATH / "sales_daily.csv")

df_sales.head()


,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


In [3]:

# Prepare sales data for customer transaction generation

df_sales["Date"] = pd.to_datetime(df_sales["Date"])

print("Sales data shape:", df_sales.shape)
print("Date range:", df_sales["Date"].min(), "to", df_sales["Date"].max())
print("Unique SKUs:", df_sales["SKU"].nunique())
print("Missing values:\n", df_sales.isnull().sum())
print("Non-positive sales rows:", (df_sales["Units_Sold"] <= 0).sum())


Sales data shape: (36550, 6)
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Unique SKUs: 50
Missing values:
 Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64
Non-positive sales rows: 202


In [5]:

# Prepare valid sales records

df_valid_sales = df_sales[
    (df_sales["Units_Sold"] > 0) &
    (df_sales["Revenue"] > 0) &
    (df_sales["Price"] > 0)
].copy()

df_valid_sales["Date"] = pd.to_datetime(df_valid_sales["Date"])

print("Valid sales records:", len(df_valid_sales))
print("Date range:", df_valid_sales["Date"].min(), "to", df_valid_sales["Date"].max())
print("Unique SKUs:", df_valid_sales["SKU"].nunique())


Valid sales records: 36348
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Unique SKUs: 50


In [7]:

# Build the sales reference for customer transaction modelling

sales_reference = (
    df_valid_sales
    .groupby(["Date", "SKU", "Price", "Promotion"], as_index=False)
    .agg(
        Total_Units=("Units_Sold", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
)

print("Reference rows:", len(sales_reference))
print("Total units:", sales_reference["Total_Units"].sum())
print("Total revenue:", round(sales_reference["Total_Revenue"].sum(), 2))
print("Unique SKUs:", sales_reference["SKU"].nunique())

sales_reference.head()


Reference rows: 36348
Total units: 511810
Total revenue: 3096056707.22
Unique SKUs: 50


,Date,SKU,Price,Promotion,Total_Units,Total_Revenue
0,2024-01-01,SKU001,3664.05,0,5,18320.25
1,2024-01-01,SKU002,3805.69,0,15,57085.35
2,2024-01-01,SKU003,8078.23,0,5,40391.15
3,2024-01-01,SKU004,6861.57,0,5,34307.85
4,2024-01-01,SKU005,9493.22,0,12,113918.64


In [9]:

# Validate product coverage across sales and inventory

df_inventory = pd.read_csv(DATA_PATH / "inventory_snapshots.csv")

sales_skus = set(sales_reference["SKU"].unique())
inventory_skus = set(df_inventory["SKU"].unique())

common_skus = sales_skus & inventory_skus

print("SKUs in sales:", len(sales_skus))
print("SKUs in inventory:", len(inventory_skus))
print("Common SKUs:", len(common_skus))
print("Sales SKUs missing from inventory:", sorted(sales_skus - inventory_skus))


SKUs in sales: 50
SKUs in inventory: 200
Common SKUs: 50
Sales SKUs missing from inventory: []


In [11]:

# Validate sales dates against the project calendar

df_calendar = pd.read_csv(DATA_PATH / "calendar.csv")
df_calendar["date"] = pd.to_datetime(df_calendar["date"])

sales_dates = set(sales_reference["Date"].dt.normalize().unique())
calendar_dates = set(df_calendar["date"].dt.normalize().unique())

print("Unique sales dates:", len(sales_dates))
print("Calendar dates:", len(calendar_dates))
print("Common dates:", len(sales_dates & calendar_dates))
print("Sales dates missing from calendar:", len(sales_dates - calendar_dates))


Unique sales dates: 731
Calendar dates: 731
Common dates: 731
Sales dates missing from calendar: 0


In [13]:

# Prepare customer profiles for segmentation

import numpy as np

np.random.seed(42)

n_customers = 500

customer_profiles = pd.DataFrame({
    "Customer_ID": [
        f"CUST{i:04d}" for i in range(1, n_customers + 1)
    ],
    "Customer_Type": np.random.choice(
        ["Regular", "Occasional", "Infrequent"],
        size=n_customers,
        p=[0.30, 0.45, 0.25]
    )
})

customer_profiles.head()


,Customer_ID,Customer_Type
0,CUST0001,Occasional
1,CUST0002,Infrequent
2,CUST0003,Occasional
3,CUST0004,Occasional
4,CUST0005,Regular


In [15]:

# Define purchase allocation weights by customer type

customer_profiles["Purchase_Weight"] = (
    customer_profiles["Customer_Type"].map({
        "Regular": 3.0,
        "Occasional": 1.5,
        "Infrequent": 0.5
    })
)

customer_profiles["Purchase_Weight"] /= (
    customer_profiles["Purchase_Weight"].sum()
)

print("Customer count:", len(customer_profiles))
print("\nCustomer type distribution:")
print(customer_profiles["Customer_Type"].value_counts())

print("\nTotal allocation weight:",
      round(customer_profiles["Purchase_Weight"].sum(), 6))


Customer count: 500

Customer type distribution:
Customer_Type
Occasional    215
Regular       154
Infrequent    131
Name: count, dtype: int64

Total allocation weight: 1.0


In [19]:

# Generate customer transaction records

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Use only valid sales records
sales = df_valid_sales.reset_index(drop=True).copy()

# Allocate each sale's units across a small number of customers
transactions = []

customer_ids = customer_profiles["Customer_ID"].to_numpy()

for row in sales.itertuples(index=False):
    units = int(row.Units_Sold)

    # Split units across 1 to 3 modelled customers
    n_parts = min(units, int(rng.integers(1, 4)))
    parts = np.ones(n_parts, dtype=int)

    remaining = units - n_parts
    if remaining > 0:
        extra = rng.multinomial(
            remaining, np.ones(n_parts) / n_parts
        )
        parts += extra

    chosen_customers = rng.choice(
        customer_ids, size=n_parts, replace=False
    )

    for customer_id, quantity in zip(chosen_customers, parts):
        transactions.append({
            "Transaction_Date": row.Date,
            "Customer_ID": customer_id,
            "SKU": row.SKU,
            "Quantity": int(quantity),
            "Unit_Price": float(row.Price),
            "Revenue": round(int(quantity) * float(row.Price), 2),
            "Country": "India",
            "Promotion": int(row.Promotion)
        })

df_customer_transactions = pd.DataFrame(transactions)

df_customer_transactions.insert(
    0,
    "Transaction_ID",
    [f"TXN{i:07d}" for i in range(1, len(df_customer_transactions) + 1)]
)

# Keep the requested column order
df_customer_transactions = df_customer_transactions[
    [
        "Transaction_ID", "Transaction_Date", "Customer_ID", "SKU",
        "Quantity", "Unit_Price", "Revenue", "Country", "Promotion"
    ]
]

DATA_PATH = Path("../data/raw")
DATA_PATH.mkdir(parents=True, exist_ok=True)

df_customer_transactions.to_csv(
    DATA_PATH / "customer_transactions.csv",
    index=False
)

print("File saved successfully!")
print("Location:", DATA_PATH / "customer_transactions.csv")
print("Rows:", len(df_customer_transactions))

df_customer_transactions.head(10)


PermissionError: [Errno 13] Permission denied: '..\\data\\raw\\customer_transactions.csv'

In [21]:

from pathlib import Path

DATA_PATH = Path("../data/raw")
output_file = DATA_PATH / "customer_transactions.csv"

print("Save location:", output_file.resolve())
print("Folder exists:", DATA_PATH.exists())
print("File exists:", output_file.exists())


Save location: C:\Users\nikam\Desktop\RetailPulse\data\raw\customer_transactions.csv
Folder exists: True
File exists: True


In [23]:

from pathlib import Path

output_file = Path(
    r"C:\Users\nikam\Desktop\RetailPulse\data\raw\customer_transactions.csv"
)

try:
    df_customer_transactions.to_csv(output_file, index=False)
    print("Data successfully saved in customer_transactions.csv")
    print("Total rows:", len(df_customer_transactions))
    print("Columns:", list(df_customer_transactions.columns))

except PermissionError:
    print("File is locked or write permission is denied.")
    print("Close Excel or any program using customer_transactions.csv, then retry.")


Data successfully saved in customer_transactions.csv
Total rows: 71661
Columns: ['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'SKU', 'Quantity', 'Unit_Price', 'Revenue', 'Country', 'Promotion']


In [25]:

# Validate the saved customer transaction data

df_check = pd.read_csv(
    r"C:\Users\nikam\Desktop\RetailPulse\data\raw\customer_transactions.csv"
)

print("Shape:", df_check.shape)
print("Missing values:", df_check.isnull().sum().sum())
print("Duplicate Transaction IDs:", df_check["Transaction_ID"].duplicated().sum())
print("Unique customers:", df_check["Customer_ID"].nunique())
print("Unique SKUs:", df_check["SKU"].nunique())
print("Date range:", df_check["Transaction_Date"].min(),
      "to", df_check["Transaction_Date"].max())

df_check.head()


Shape: (71661, 9)
Missing values: 0
Duplicate Transaction IDs: 0
Unique customers: 500
Unique SKUs: 50
Date range: 2024-01-01 to 2025-12-31


,Transaction_ID,Transaction_Date,Customer_ID,SKU,Quantity,Unit_Price,Revenue,Country,Promotion
0,TXN0000001,2024-01-01,CUST0387,SKU001,5,3664.05,18320.25,India,0
1,TXN0000002,2024-01-01,CUST0220,SKU002,9,3805.69,34251.21,India,0
2,TXN0000003,2024-01-01,CUST0043,SKU002,6,3805.69,22834.14,India,0
3,TXN0000004,2024-01-01,CUST0048,SKU003,5,8078.23,40391.15,India,0
4,TXN0000005,2024-01-01,CUST0487,SKU004,3,6861.57,20584.71,India,0


In [27]:

# Compare original sales totals with customer transaction totals

print("Original total units:", df_valid_sales["Units_Sold"].sum())
print("Customer transaction units:", df_check["Quantity"].sum())

print("\nOriginal total revenue:",
      round(df_valid_sales["Revenue"].sum(), 2))
print("Customer transaction revenue:",
      round(df_check["Revenue"].sum(), 2))

print("\nUnits match:",
      df_valid_sales["Units_Sold"].sum() == df_check["Quantity"].sum())

print("Revenue matches:",
      round(df_valid_sales["Revenue"].sum(), 2)
      == round(df_check["Revenue"].sum(), 2))


Original total units: 511810
Customer transaction units: 511810

Original total revenue: 3096056707.22
Customer transaction revenue: 3096056707.22

Units match: True
Revenue matches: True
